In [1]:
# === Install library ===
!pip install Sastrawi tqdm

import pandas as pd
import string
import nltk
import requests
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from google.colab import files
from functools import lru_cache
from tqdm.notebook import tqdm

# Download resource NLTK
nltk.download('punkt')
nltk.download('punkt_tab')   # fix error punkt_tab
nltk.download('wordnet')

# Tools
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()
factory_stop = StopWordRemoverFactory()
stopword = factory_stop.create_stop_word_remover()
lemmatizer = WordNetLemmatizer()

# Wordlist Indonesia
url = "https://raw.githubusercontent.com/damarpayung/indonesian-wordlist/master/indonesian.txt"
response = requests.get(url)
indonesian_dict = set(response.text.splitlines()) if response.status_code == 200 else set()

def pembakuan_kata(token):
    return token if token in indonesian_dict else token

# === Caching untuk stemming ===
@lru_cache(maxsize=50000)
def cached_stem(word):
    return stemmer.stem(word)

# Fungsi preprocessing
def preprocess(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = stopword.remove(text)
    tokens = word_tokenize(text)
    tokens = [pembakuan_kata(t) for t in tokens]
    tokens = [cached_stem(t) for t in tokens]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

# Upload file
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Terapkan preprocessing dengan progress bar
for col in df.select_dtypes(include=['object']).columns:
    tqdm.pandas(desc=f"Processing {col}")   # progress bar per kolom
    df[col] = df[col].astype(str).progress_apply(preprocess)

# Simpan hasil
output_file = "PPW_HasilCrawling_FakultasTeknik_preprocessed.csv"
df.to_csv(output_file, index=False)
files.download(output_file)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 13.7 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Saving PPW_HasilCrawling_FakultasTeknik_partial.csv to PPW_HasilCrawling_FakultasTeknik_partial.csv


Processing penulis:   0%|          | 0/2298 [00:00<?, ?it/s]

Processing judul:   0%|          | 0/2298 [00:00<?, ?it/s]

...

Processing abstrak_bindonesia:   0%|          | 0/2298 [00:00<?, ?it/s]

Processing abstrak_binggris:   0%|          | 0/2298 [00:00<?, ?it/s]

Processing nama_prodi:   0%|          | 0/2298 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>